# Fashion Similarity Model Training

Train a custom deep learning model on DeepFashion dataset for accurate fashion product similarity matching.

**Hardware:**
- GPU: RTX 4060 8GB ✅
- CUDA: 13.0 ✅
- RAM: 32GB ✅

**Expected Time:** 45 min - 2 hours (depending on dataset size)

## 1. Setup & Imports

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import matplotlib.pyplot as plt
import os
import json
from pathlib import Path

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

## 2. GPU Configuration

In [ ]:
# Check GPU availability
print("\n🔍 GPU Configuration:")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs Available: {gpus}")

if gpus:
    try:
        # Enable memory growth
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ GPU memory growth enabled for {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(f"⚠️ GPU setup warning: {e}")
else:
    print("⚠️ No GPU detected! Training will be very slow.")

## 3. Configuration

**📝 IMPORTANT:** Update these paths to match your system!

In [ ]:
# ========================================
# CONFIGURATION - UPDATE THESE PATHS!
# ========================================

# Dataset paths
DATASET_ROOT = r"D:\My_PaidProjects\Lufyco_Clothing\DeepFashion"
TRAIN_DIR = os.path.join(DATASET_ROOT, "train_images")

# Model settings
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32  # Reduce to 16 if you get OOM errors
EPOCHS = 20
LEARNING_RATE = 0.0001
VALIDATION_SPLIT = 0.2

# Output paths
OUTPUT_DIR = r"D:\My_PaidProjects\Lufyco_Clothing\Lufyco_Backend\models"
MODEL_NAME = "fashion-similarity-model"

# Verify paths
print(f"\n📂 Dataset directory: {TRAIN_DIR}")
print(f"📂 Output directory: {OUTPUT_DIR}")
print(f"✅ Dataset exists: {os.path.exists(TRAIN_DIR)}")

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 4. Data Preparation

Load and augment training data

In [ ]:
# Training data with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    fill_mode='nearest',
    validation_split=VALIDATION_SPLIT
)

# Validation data (no augmentation)
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=VALIDATION_SPLIT
)

# Create generators
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

validation_generator = val_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

print(f"\n✅ Training samples: {train_generator.samples}")
print(f"✅ Validation samples: {validation_generator.samples}")
print(f"\n📊 Classes found: {list(train_generator.class_indices.keys())}")
print(f"📊 Number of classes: {len(train_generator.class_indices)}")

## 5. Visualize Sample Images

In [ ]:
# Display sample images from training set
sample_batch, sample_labels = next(train_generator)

plt.figure(figsize=(12, 8))
for i in range(min(9, len(sample_batch))):
    plt.subplot(3, 3, i + 1)
    plt.imshow(sample_batch[i])
    class_idx = np.argmax(sample_labels[i])
    class_name = list(train_generator.class_indices.keys())[class_idx]
    plt.title(f"Class: {class_name}")
    plt.axis('off')

plt.tight_layout()
plt.show()

print(f"\n📸 Displaying {min(9, len(sample_batch))} sample images")

## 6. Build Model Architecture

Using **MobileNetV2** with transfer learning

In [ ]:
# Load pre-trained MobileNetV2
base_model = MobileNetV2(
    input_shape=(*IMAGE_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze base model
base_model.trainable = False
print(f"✅ Loaded MobileNetV2 (frozen {len(base_model.layers)} layers)")

# Build complete model
num_classes = len(train_generator.class_indices)

model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu', name='dense_1'),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu', name='dense_2'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax', name='predictions')
], name='Fashion_Similarity_Model')

# Compile
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_accuracy')]
)

model.summary()

## 7. Setup Callbacks

In [ ]:
# Callbacks for training
callbacks = [
    # Save best model
    keras.callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_DIR, f"{MODEL_NAME}_best.h5"),
        save_best_only=True,
        monitor='val_accuracy',
        mode='max',
        verbose=1
    ),
    
    # Early stopping
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    
    # Reduce learning rate on plateau
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

print("✅ Callbacks configured")

## 8. Train Model 🏋️

⚠️ **This will take some time!** Grab a coffee ☕

In [ ]:
# Train the model
print("\n🏋️ Starting training...\n")

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Training complete!")

## 9. Visualize Training Results

In [ ]:
# Plot training history
plt.figure(figsize=(14, 5))

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Print final metrics
print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
print(f"Training Accuracy:   {history.history['accuracy'][-1]*100:.2f}%")
print(f"Validation Accuracy: {history.history['val_accuracy'][-1]*100:.2f}%")
print(f"Training Loss:       {history.history['loss'][-1]:.4f}")
print(f"Validation Loss:     {history.history['val_loss'][-1]:.4f}")
print("="*60)

## 10. Save Model for Production

In [ ]:
# Save TensorFlow model
model_path = os.path.join(OUTPUT_DIR, f"{MODEL_NAME}.h5")
model.save(model_path)
print(f"✅ Saved TensorFlow model: {model_path}")

# Save for TensorFlow.js (Node.js backend)
try:
    import tensorflowjs as tfjs
    tfjs_path = os.path.join(OUTPUT_DIR, f"{MODEL_NAME}_tfjs")
    os.makedirs(tfjs_path, exist_ok=True)
    tfjs.converters.save_keras_model(model, tfjs_path)
    print(f"✅ Saved TensorFlow.js model: {tfjs_path}")
except ImportError:
    print("⚠️ tensorflowjs not installed")
    print("   Install: pip install tensorflowjs")

# Save class indices
class_indices_path = os.path.join(OUTPUT_DIR, "class_indices.json")
with open(class_indices_path, 'w') as f:
    json.dump(train_generator.class_indices, f, indent=2)
print(f"✅ Saved class indices: {class_indices_path}")

## 11. Test Model Predictions

In [ ]:
# Get test batch
test_batch, test_labels = next(validation_generator)

# Make predictions
predictions = model.predict(test_batch)

# Display results
plt.figure(figsize=(15, 10))
for i in range(min(6, len(test_batch))):
    plt.subplot(2, 3, i + 1)
    plt.imshow(test_batch[i])
    
    # True label
    true_idx = np.argmax(test_labels[i])
    true_class = list(train_generator.class_indices.keys())[true_idx]
    
    # Predicted label
    pred_idx = np.argmax(predictions[i])
    pred_class = list(train_generator.class_indices.keys())[pred_idx]
    confidence = predictions[i][pred_idx] * 100
    
    # Color: green if correct, red if wrong
    color = 'green' if true_idx == pred_idx else 'red'
    
    plt.title(f"True: {true_class}\nPred: {pred_class} ({confidence:.1f}%)", color=color)
    plt.axis('off')

plt.tight_layout()
plt.show()

## 🎉 Training Complete!

### Next Steps:
1. Restart your Node.js backend
2. The model will load automatically
3. Test with `/api/ai/image-search` endpoint

### Model Location:
- TensorFlow: `{OUTPUT_DIR}/{MODEL_NAME}.h5`
- TensorFlow.js: `{OUTPUT_DIR}/{MODEL_NAME}_tfjs/`
- Class indices: `{OUTPUT_DIR}/class_indices.json`